# Legal Retrieval Module Evaluation

This notebook evaluates the `LegalRetrievalModule` in combination with the `TriageModule` and `SemanticRouterModule`. It runs through the evaluation dataset and appends the outputs of all three modules.

In [7]:
%pip install pandas openpyxl tqdm requests python-dotenv rich faiss-cpu

You should consider upgrading via the '/Users/sebastianrafaellachica/codingprojects/Legal-Adaptive-Routing-Framework/myvenv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import sys
import os
import time
import json
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import modules
from src.adaptive_routing import FrameworkConfig, TriageModule, SemanticRouterModule, LegalRetrievalModule

# Load environment variables
load_dotenv(os.path.join(project_root, '.env'))

print("Environment setup complete.")

Environment setup complete.


In [4]:
# Initialize configuration
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
FrameworkConfig._update_settings_(api_key=OPENROUTER_API_KEY)

# Initialize modules
print("Initializing Triage Module...")
triage_module = TriageModule()

print("Initializing Semantic Router Module...")
router_module = SemanticRouterModule()

print("Initializing Legal Retrieval Module...")
retrieval_module = LegalRetrievalModule()

# Load FAISS index
index_dir = os.path.join(project_root, "localfiles", "legal-basis")
index_file = os.path.join(index_dir, "combined_index.faiss")
chunks_file = os.path.join(index_dir, "combined_index.json")

if os.path.exists(index_file) and os.path.exists(chunks_file):
    print("Loading existing FAISS index...")
    retrieval_module._load_index_(index_file, chunks_file)
    print("FAISS index loaded successfully.")
else:
    print(f"ERROR: FAISS index not found in {index_dir}. Please run index builder first.")


Initializing Triage Module...
Initializing Semantic Router Module...
Initializing Legal Retrieval Module...
Loading existing FAISS index...
FAISS index loaded successfully.


In [9]:
# Path to the evaluation dataset
dataset_path = 'dataset/Legal-Retrieval-Eval-Dataset.xlsx'

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

# Load the dataset
df = pd.read_excel(dataset_path)

print(f"Loaded dataset with {len(df)} rows.")
print("Columns:", df.columns.tolist())

df.head()

Loaded dataset with 153 rows.
Columns: ['test_id', 'query', 'Source', 'ground_truth_corpus', 'target_article', 'premise', 'hypothesis']


,test_id,query,Source,ground_truth_corpus,target_article,premise,hypothesis
0,1.0,"""I worked for 6 years and my boss died. Do I g...",HK Laws,"Employment Ordinance (Chapter 57), Laws of Hon...",Section 31V(1)(a),"If an employer dies, a worker with 5+ years se...",You can claim a long service payment because y...
1,2.0,"""My holiday is on my day off. Do I lose it?""",HK Laws,"Employment Ordinance (Chapter 57), Laws of Hon...",Section 39(4),"If a statutory holiday falls on a rest day, th...",If your rest day and a holiday land on the sam...
2,3.0,"""Can I quit if my boss hit me?""",PH Republic Act,Migrant Workers and Overseas Filipinos Act (RA...,Section 10 (Just Cause),OFWs can terminate employment without notice f...,You can leave immediately without notice and s...
3,4.0,"""The ship is in a war zone. Can I say no?""",PH Republic Act,Magna Carta of Filipino Seafarers (RA 12021),Section 34 (Right to Refusal),Seafarers have the right to refuse to sail int...,You are allowed to refuse to stay on the ship ...
4,5.0,"""I was sick for 5 days. Do I get paid?""",HK Laws,"Employment Ordinance (Chapter 57), Laws of Hon...",Section 33(3)(a),Sickness allowance is paid if the sick leave i...,If you have a doctor's note for 5 straight day...


In [10]:
# Define columns to append
cols_to_add = [
    'triage_normalized_text', 
    'triage_detected_language', 
    'router_route', 
    'router_search_signals', 
    'retrieval_dominant_corpus', 
    'retrieval_chunks'
]

for col in cols_to_add:
    if col not in df.columns:
        df[col] = None

# Evaluation Loop Configuration
max_retries = 3
retry_delay = 3  # seconds
checkpoint_path = 'dataset/Legal-Retrieval-Eval-Checkpoint.xlsx'

print("Starting evaluation processing...")

success_count = 0
for index, row in tqdm(df.iterrows(), total=len(df)):
    # Skip already processed rows to support checkpoint resuming
    if pd.notna(row['router_route']) and row['router_route'] != "ERROR":
        continue

    raw_query = row['query']
    if pd.isna(raw_query) or str(raw_query).strip() == "":
        continue

    attempt = 0
    while attempt < max_retries:
        try:
            # 1. Triage
            triaged_data = triage_module._process_request_(str(raw_query))
            normalized_text = triaged_data.get("normalized_text", str(raw_query))
            detected_lang = triaged_data.get("detected_language", "Unknown")
            
            # 2. Router
            classification = router_module._process_routing_(normalized_text, history=None, threshold=0.1)
            route = classification.get("route", "General-LLM")
            signals = classification.get("search_signals")
            
            # 3. Retrieval
            dominant_corpus = None
            chunks_json = None
            
            if signals is not None and route != "Casual-LLM":
                retrieval_output = retrieval_module._process_retrieval_(normalized_text, signals=signals)
                retrieved_chunks = retrieval_output.get("retrieved_chunks", [])
                dominant_corpus = retrieval_output.get("dominant_corpus")
                
                # Format chunks for saving
                formatted_chunks = [
                    {
                        "source": c.get("source"),
                        "text": c.get("chunk", "")[:200] + "...", # truncate to save space
                        "score": float(c.get("score", 0.0))
                    } for c in retrieved_chunks
                ]
                chunks_json = json.dumps(formatted_chunks)
            
            # Update DataFrame
            df.at[index, 'triage_normalized_text'] = normalized_text
            df.at[index, 'triage_detected_language'] = detected_lang
            df.at[index, 'router_route'] = route
            df.at[index, 'router_search_signals'] = json.dumps(signals) if signals else None
            df.at[index, 'retrieval_dominant_corpus'] = dominant_corpus
            df.at[index, 'retrieval_chunks'] = chunks_json
            
            success_count += 1
            
            # Autosave
            if success_count % 5 == 0:
                df.to_excel(checkpoint_path, index=False)
                
            break # Success, exit retry loop
            
        except Exception as e:
            attempt += 1
            if attempt < max_retries:
                time.sleep(retry_delay)
            else:
                print(f"Failed row {index} after {max_retries} attempts: {e}")
                df.at[index, 'router_route'] = "ERROR"

print("Processing complete.")
df.to_excel(checkpoint_path, index=False) # Final checkpoint


Starting evaluation processing...


  0%|          | 0/153 [00:00<?, ?it/s]Language tag not found in normalizer output. First 100 chars: To provide a professional legal linguistic normalization, I must first categorize your inquiry. Your
JSON parsing failed. Attempting strict regex fallback extraction...
Failed to parse router output as JSON and regex fallback failed. Raw output: "This is an excellent and thorough normalization of a common layperson's query. You've effectively broken down a complex situation into understandable legal frameworks and identified the crucial inform"
JSON parsing failed. Attempting strict regex fallback extraction...
Failed to parse router output as JSON and regex fallback failed. Raw output: "This is an excellent and thorough normalization of a common layperson's query. You've effectively broken down a complex situation into understandable legal frameworks and identified the crucial inform"
JSON parsing failed. Attempting strict regex fallback extraction...
Failed to parse router output as J

KeyboardInterrupt: 

## Automated Evaluation Checker
Calculate Information Retrieval metrics (Accuracy/Hit Rate, Precision, Recall, F1-Score).

In [ ]:
# Function to check if target article is in retrieved chunks
def calculate_metrics(df):
    total_queries = 0
    hits = 0 # accuracy
    total_precision = 0.0
    total_recall = 0.0
    
    # We will also record per-query hit status
    if 'eval_is_hit' not in df.columns:
        df['eval_is_hit'] = False
        
    for index, row in df.iterrows():
        if pd.isna(row['query']) or pd.isna(row['target_article']):
            continue
            
        total_queries += 1
        target_article = str(row['target_article']).lower()
        chunks_json_str = row['retrieval_chunks']
        
        relevant_retrieved = 0
        total_retrieved = 0
        
        if pd.notna(chunks_json_str) and chunks_json_str != "null":
            try:
                chunks = json.loads(chunks_json_str)
                total_retrieved = len(chunks)
                
                # Check for hits in metadata 'source' or chunk 'text'
                for chunk in chunks:
                    source = str(chunk.get('source', '')).lower()
                    text = str(chunk.get('text', '')).lower()
                    
                    if target_article in source or target_article in text:
                        relevant_retrieved += 1
            except Exception as e:
                pass
                
        # Calculate per query metrics
        # Recall@K = 1 if the target article was found at least once, else 0
        query_hit = 1 if relevant_retrieved > 0 else 0
        
        # Precision@K = (relevant retrieved) / total retrieved
        query_precision = (relevant_retrieved / total_retrieved) if total_retrieved > 0 else 0.0
        
        df.at[index, 'eval_is_hit'] = bool(query_hit)
        
        hits += query_hit
        total_precision += query_precision
        total_recall += query_hit # since total relevant per query is exactly 1 (the target_article)

    # Aggregate metrics
    accuracy = (hits / total_queries) if total_queries > 0 else 0.0
    avg_precision = (total_precision / total_queries) if total_queries > 0 else 0.0
    avg_recall = (total_recall / total_queries) if total_queries > 0 else 0.0
    
    if (avg_precision + avg_recall) > 0:
        f1_score = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall)
    else:
        f1_score = 0.0
        
    return {
        "Total Queries Evaluated": total_queries,
        "Accuracy (Hit Rate)": f"{accuracy * 100:.2f}%",
        "Average Precision": f"{avg_precision:.4f}",
        "Average Recall": f"{avg_recall:.4f}",
        "F1-Score": f"{f1_score:.4f}"
    }

metrics = calculate_metrics(df)

import rich
rich.print("[bold green]Evaluation Results:[/bold green]")
for k, v in metrics.items():
    rich.print(f"[bold]{k}:[/bold] {v}")


In [ ]:
# Save final results
output_path = 'dataset/Legal-Retrieval-Eval-Results.xlsx'
df.to_excel(output_path, index=False)

print(f"Final results saved to: {output_path}")

# Display a sample of hits vs misses
df[['query', 'target_article', 'eval_is_hit']].head(10)
